# Option Portfolio Metrics

Loads **your specific option positions** from `data/option_data.csv`, enriches them with
**fresh implied volatility** interpolated from the downloaded market chain (SQLite DB),
then computes Greeks, probabilities and payoff for those positions only.

The full downloaded chain is used purely as an IV surface reference — it is never directly analysed.

## 1 · Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

from skew import (
    compute_option_metrics,
    portfolio_greeks,
    market_implied_pdf,
    prob_profit_from_pdf,
    load_option_snapshots,
    fit_svi_slice,
    eval_svi_iv,
    bs_price_forward,
    bs_greeks_full,
)

In [ ]:
# Auto-detect project root
_cwd = Path(os.getcwd())
if   (_cwd / "skew").exists():        _project_root = _cwd
elif (_cwd.parent / "skew").exists(): _project_root = _cwd.parent
else:                                  _project_root = _cwd

PORTFOLIO_CSV = str(_project_root / "data" / "option_data.csv")
DB_PATH       = str(_project_root / "data" / "options.db")

# None = use the most recent snapshot in the DB for IV interpolation
SNAPSHOT_DATE = None
DEFAULT_RF    = 0.04
CONTRACT_SIZE = 100     # shares per contract

print(f"Portfolio : {PORTFOLIO_CSV}")
print(f"Market DB : {DB_PATH}")

In [ ]:
## 2 · Load portfolio positions (minimal CSV)

# Only required columns: symbol, option_type, strike, expiry
# Optional: contracts (defaults to 1)
pf_raw = pd.read_csv(PORTFOLIO_CSV, usecols=lambda c: c in
    {"symbol", "option_type", "strike", "expiry", "contracts"})

pf = pf_raw.copy()

# Drop rows where required fields are missing
pf = pf.dropna(subset=["symbol", "option_type", "strike", "expiry"])

# Ensure string dtype (guards against mixed-type columns)
pf["symbol"] = pf["symbol"].astype(str).str.strip().str.upper()

# option_type → "C" / "P"
pf["option_type"] = (
    pf["option_type"].astype(str).str.upper().str.strip()
    .replace({"CALL": "C", "PUT": "P"})
)

# expiry as datetime (handles dd/mm/yyyy, yyyy-mm-dd, etc.)
pf["expiry"] = pd.to_datetime(pf["expiry"], dayfirst=True, errors="coerce")

# Drop rows where expiry couldn't be parsed
pf = pf.dropna(subset=["expiry"])

# T (years to expiry from today)
pf["T"] = ((pf["expiry"] - pd.Timestamp.today().normalize()).dt.days / 365.0).clip(lower=1e-6)

# contracts default
if "contracts" not in pf.columns:
    pf["contracts"] = 1

pf = pf.reset_index(drop=True)

print(f"Loaded {len(pf)} portfolio positions  |  tickers: {sorted(pf['symbol'].unique())}")
pf

In [ ]:
## 3 · Enrich each position from market chain (spot, IV, bid, ask, forward …)

def _load_chain(ticker, db_path, snapshot_date):
    raw = load_option_snapshots(ticker, db_path=db_path, snapshot_date=snapshot_date)
    if raw.empty:
        return pd.DataFrame()
    snap = snapshot_date or raw["snapshot_date"].max()
    df = raw[raw["snapshot_date"] == snap].copy()
    df["expiry"] = pd.to_datetime(df["expiry"])
    df["option_type"] = (
        df["option_type"].str.upper().str.strip()
        .replace({"CALL": "C", "PUT": "P"})
    )
    return df


def _svi_iv_for_strike(chain_exp, forward, T_exp, target_strike):
    """
    Fit SVI to OTM options in chain_exp and evaluate at target_strike.
    Falls back to np.interp on the raw slice if the fit fails.
    """
    F = forward
    calls_otm = chain_exp[(chain_exp["option_type"] == "C") & (chain_exp["strike"] >= F)]
    puts_otm  = chain_exp[(chain_exp["option_type"] == "P") & (chain_exp["strike"] <  F)]
    otm = pd.concat([puts_otm, calls_otm]).dropna(subset=["implied_vol"]).sort_values("strike")

    if len(otm) >= 5:
        params = fit_svi_slice(otm["strike"].values, otm["implied_vol"].values, F, T_exp)
        if params is not None:
            return float(eval_svi_iv(params, [target_strike], F, T_exp)[0])

    # Fallback: linear interpolation (no extrapolation beyond range)
    k_obs = chain_exp["strike"].values
    iv_obs = pd.to_numeric(chain_exp["implied_vol"], errors="coerce").values
    return float(np.interp(target_strike, k_obs, iv_obs))


def enrich_from_chain(ticker, option_type, expiry, strike, chain_cache):
    """Return a dict of market fields for a portfolio position."""
    empty = dict(spot=np.nan, implied_vol=np.nan, lastPrice=np.nan,
                 bid=np.nan, ask=np.nan, div_yield=0.0,
                 disc_factor=1.0, forward=np.nan)

    chain = chain_cache.get(ticker.upper(), pd.DataFrame())
    if chain.empty:
        return empty

    # closest expiry in chain
    exp_dt = pd.to_datetime(expiry)
    available = pd.to_datetime(chain["expiry"].unique())
    closest_exp = available[np.argmin(np.abs(available - exp_dt))]

    # all options at this expiry (for scalar fields)
    sl_all  = chain[chain["expiry"] == closest_exp].sort_values("strike")
    # same option type at this expiry (for bid/ask interp)
    sl_type = sl_all[sl_all["option_type"] == option_type.upper()]
    sl_ref  = sl_type if len(sl_type) >= 2 else sl_all

    if sl_all.empty:
        return empty

    spot      = float(sl_all["underlying_price"].iloc[0])
    div_yield = float(sl_all["div_yield"].iloc[0]) if "div_yield" in sl_all.columns else 0.0
    disc_fac  = float(sl_all["disc_factor"].iloc[0]) if "disc_factor" in sl_all.columns else 1.0
    forward   = float(sl_all["forward"].iloc[0]) if "forward" in sl_all.columns else spot
    T_exp     = float(sl_all["T"].iloc[0]) if "T" in sl_all.columns else max((closest_exp - pd.Timestamp.today()).days / 365.0, 1e-6)

    def _interp(sl, col):
        if col not in sl.columns:
            return np.nan
        vals = pd.to_numeric(sl[col], errors="coerce").values
        return float(np.interp(strike, sl["strike"].values, vals))

    bid = _interp(sl_ref, "bid")
    ask = _interp(sl_ref, "ask")
    mid = (0.5 * (bid + ask) if np.isfinite(bid) and np.isfinite(ask)
           else max(x for x in (bid, ask) if np.isfinite(x)) if any(np.isfinite(x) for x in (bid, ask))
           else np.nan)

    # ── SVI-fitted IV (interpolates + extrapolates convexly) ──────────────────
    iv = _svi_iv_for_strike(sl_all, forward, T_exp, strike)

    return dict(spot=spot, implied_vol=iv, lastPrice=mid,
                bid=bid, ask=ask, div_yield=div_yield,
                disc_factor=disc_fac, forward=forward)


# ── Build chain cache ─────────────────────────────────────────────────────────
tickers_needed = pf["symbol"].str.upper().unique()
chain_cache = {}
for tkr in tickers_needed:
    chain_cache[tkr] = _load_chain(tkr, DB_PATH, SNAPSHOT_DATE)
    n = len(chain_cache[tkr])
    snap = chain_cache[tkr]["snapshot_date"].max() if n else "N/A"
    print(f"  {tkr}: {n} chain rows  |  snapshot={snap}" if n
          else f"  {tkr}: NOT FOUND in DB — Greeks will use NaN IV")

# ── Enrich portfolio ──────────────────────────────────────────────────────────
enriched_rows = []
for _, row in pf.iterrows():
    market = enrich_from_chain(row["symbol"], row["option_type"],
                               row["expiry"], row["strike"], chain_cache)
    enriched_rows.append({**row.to_dict(), **market})

pf_enriched = pd.DataFrame(enriched_rows)
pf_enriched["r"] = DEFAULT_RF

n_iv = pf_enriched["implied_vol"].notna().sum()
print(f"\nMarket data enriched: {n_iv}/{len(pf_enriched)} positions have IV")
pf_enriched[["symbol","option_type","strike","expiry","spot","implied_vol","lastPrice","bid","ask","div_yield"]].round(4)

In [ ]:
## 4 · Compute Greeks & probabilities

metrics = compute_option_metrics(pf_enriched, default_rf=DEFAULT_RF, contract_size=CONTRACT_SIZE)

display_cols = [
    "symbol", "option_type", "strike", "expiry", "T", "implied_vol",
    "spot", "intrinsic", "extrinsic", "moneyness", "break_even", "max_loss",
    "prob_itm", "prob_profit", "prob_profit_ask",
    "delta", "gamma", "vega", "theta_day", "rho",
]
available = [c for c in display_cols if c in metrics.columns]
metrics[available].round(4)

In [ ]:
## 5 · Portfolio-level Greeks — summary, per-leg bars, and profiles vs spot

port = portfolio_greeks(metrics)

# ── Summary table ─────────────────────────────────────────────────────────────
_g_map = {
    "delta":     "Delta",
    "gamma":     "Gamma",
    "vega":      "Vega (per 1% IV move)",
    "theta_day": "Theta (per day)",
    "rho":       "Rho (per 1% rate move)",
}
display(
    pd.DataFrame(
        [{"Greek": lbl, "Net Position": round(port.get(g, np.nan), 4)}
         for g, lbl in _g_map.items()]
    ).set_index("Greek")
)

# ── Per-leg contribution bar chart ────────────────────────────────────────────
leg_labels = (
    metrics["symbol"] + " " + metrics["option_type"] +
    " K=" + metrics["strike"].astype(str) +
    " " + metrics["expiry"].dt.strftime("%b%y")
).tolist()

_greek_bars = [
    ("pos_delta",     "Delta",                "steelblue"),
    ("pos_gamma",     "Gamma",                "seagreen"),
    ("pos_vega",      "Vega (per 1% IV)",     "orange"),
    ("pos_theta_day", "Theta (per day)",      "tomato"),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
x = np.arange(len(metrics))
for ax, (col, title, color) in zip(axes.flat, _greek_bars):
    if col not in metrics.columns:
        ax.set_visible(False); continue
    vals = metrics[col].values
    bar_colors = [color if v >= 0 else "salmon" for v in vals]
    ax.bar(x, vals, color=bar_colors)
    ax.axhline(0, color="black", lw=0.8)
    net = float(np.nansum(vals))
    ax.axhline(net, color="navy", lw=1.5, linestyle="--", label=f"Net: {net:.4g}")
    ax.set_xticks(x)
    ax.set_xticklabels(leg_labels, rotation=30, ha="right", fontsize=8)
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)
plt.suptitle("Portfolio Greeks — Per-Position Contribution", fontsize=12)
plt.tight_layout()
plt.show()

# ── Greek profiles vs spot price ──────────────────────────────────────────────
for exp in sorted(metrics["expiry"].dropna().unique()):
    legs = metrics[metrics["expiry"] == exp]
    if legs.empty:
        continue

    spot_val = float(legs["spot"].iloc[0])
    S_range  = np.linspace(spot_val * 0.55, spot_val * 1.50, 300)

    port_delta = np.zeros(len(S_range))
    port_gamma = np.zeros(len(S_range))
    port_vega  = np.zeros(len(S_range))
    port_theta = np.zeros(len(S_range))

    for _, leg in legs.iterrows():
        K      = float(leg["strike"])
        qty    = float(leg.get("contracts", 1)) * CONTRACT_SIZE
        T_leg  = max(float(leg["T"]), 1e-6)
        sigma  = float(leg["implied_vol"]) if pd.notna(leg.get("implied_vol")) else 0.0
        r      = float(leg.get("r", DEFAULT_RF))
        q      = float(leg.get("div_yield", 0.0))
        otype  = leg["option_type"]
        for j, S in enumerate(S_range):
            g = bs_greeks_full(otype, S, K, T_leg, r, q, sigma)
            port_delta[j] += qty * g.get("delta",     0.0)
            port_gamma[j] += qty * g.get("gamma",     0.0)
            port_vega[j]  += qty * g.get("vega",      0.0)
            port_theta[j] += qty * g.get("theta_day", 0.0)

    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
    _greek_lines = [
        (port_delta, "Delta",                "steelblue"),
        (port_gamma, "Gamma",                "seagreen"),
        (port_vega,  "Vega (per 1% IV)",     "orange"),
        (port_theta, "Theta (per day)",      "tomato"),
    ]
    for ax, (arr, label, color) in zip(axes.flat, _greek_lines):
        ax.plot(S_range, arr, lw=2, color=color)
        ax.axhline(0, color="black", lw=0.8)
        ax.axvline(spot_val, color="grey", lw=1, linestyle="--", alpha=0.7,
                   label=f"Spot {spot_val:.1f}")
        for _, leg in legs.iterrows():
            ax.axvline(leg["strike"], color="purple", lw=0.6, linestyle=":", alpha=0.5)
        ax.fill_between(S_range, arr, 0, where=(arr > 0), alpha=0.12, color="green")
        ax.fill_between(S_range, arr, 0, where=(arr < 0), alpha=0.12, color="red")
        ax.set_ylabel(label); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    for ax in axes[1]:
        ax.set_xlabel("Underlying price")
    plt.suptitle(
        f"Greek Profiles vs Spot — expiry {pd.Timestamp(exp).strftime('%d %b %Y')}",
        fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
## 6 · Probability summary — per position
#
# Three probabilities are shown for each option:
#   P(ITM)          → pure risk-neutral probability the option expires in-the-money
#                     (ignores the premium you paid — this is just N(d2))
#   P(profit @ mid) → probability the underlying moves far enough to cover the mid-price premium
#                     i.e. P(S_T > K + mid) for calls, P(S_T < K - mid) for puts
#   P(profit @ ask) → same but using the ask price — worst-case fill, hardest break-even to reach
#
# The gap between P(ITM) and P(profit@ask) shows how much the premium erodes your edge.

labels = (
    metrics["symbol"] + " " + metrics["option_type"] +
    " K=" + metrics["strike"].astype(str) +
    " " + metrics["expiry"].dt.strftime("%b%y")
).tolist()

n = len(metrics)
y = np.arange(n)
h = 0.25

fig, ax = plt.subplots(figsize=(11, max(4, n * 0.9)))

b1 = ax.barh(y + h,  metrics["prob_itm"],       height=h, color="steelblue", alpha=0.85,
             label="P(ITM)  — expires in-the-money, ignores premium")
b2 = ax.barh(y,      metrics["prob_profit"],     height=h, color="seagreen",  alpha=0.85,
             label="P(profit @ mid)  — covers mid-price entry cost")
b3 = ax.barh(y - h,  metrics["prob_profit_ask"], height=h, color="orange",    alpha=0.85,
             label="P(profit @ ask)  — covers ask-price entry cost (conservative)")

# Percentage labels on each bar
for bars in [b1, b2, b3]:
    for bar in bars:
        w = bar.get_width()
        if np.isfinite(w) and w > 0.01:
            ax.text(w + 0.012, bar.get_y() + bar.get_height() / 2,
                    f"{w:.0%}", va="center", ha="left", fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlim(0, 1.18)
ax.axvline(0.5, color="black", lw=1, linestyle="--", alpha=0.35, label="50% line")
ax.set_xlabel("Probability")
ax.set_title("Probability of Profit per Position\n"
             "P(ITM) ≥ P(profit@mid) ≥ P(profit@ask)  —  gap = premium drag on break-even",
             fontsize=10)
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# ── Colour-coded table ────────────────────────────────────────────────────────
_tbl = metrics[["symbol", "option_type", "strike", "expiry", "implied_vol",
                "break_even", "prob_itm", "prob_profit", "prob_profit_ask"]].copy()
_tbl = _tbl.rename(columns={
    "symbol": "Ticker", "option_type": "Type", "strike": "Strike",
    "expiry": "Expiry",  "implied_vol": "IV",
    "break_even":       "Break-even",
    "prob_itm":         "P(ITM)",
    "prob_profit":      "P(profit@mid)",
    "prob_profit_ask":  "P(profit@ask)",
})
display(
    _tbl.style
    .format({
        "IV":             "{:.1%}",
        "Break-even":     "{:.2f}",
        "P(ITM)":         "{:.1%}",
        "P(profit@mid)":  "{:.1%}",
        "P(profit@ask)":  "{:.1%}",
        "Expiry":         lambda x: x.strftime("%d %b %Y") if pd.notna(x) else "",
    })
    .background_gradient(
        subset=["P(ITM)", "P(profit@mid)", "P(profit@ask)"],
        cmap="RdYlGn", vmin=0.10, vmax=0.70)
    .hide(axis="index")
    .set_caption(
        "Green = higher probability  ·  Red = lower probability  ·  "
        "Break-even = underlying price needed at expiry to recover premium"
    )
)

In [ ]:
## 7 · Portfolio payoff — at expiry + today's theoretical value

def _bs_vec(is_call, S_arr, K, T, r, q, sigma):
    """Vectorized Black-Scholes price over an array of spot values."""
    from scipy.stats import norm as _norm
    if sigma <= 0 or T <= 0:
        intrinsic = np.maximum(S_arr - K, 0) if is_call else np.maximum(K - S_arr, 0)
        return intrinsic * np.exp(-r * T)
    F   = S_arr * np.exp((r - q) * T)
    D   = np.exp(-r * T)
    vol = sigma * np.sqrt(T)
    d1  = np.log(F / K) / vol + 0.5 * vol
    d2  = d1 - vol
    if is_call:
        return D * (F * _norm.cdf(d1) - K * _norm.cdf(d2))
    else:
        return D * (K * _norm.cdf(-d2) - F * _norm.cdf(-d1))


for exp in sorted(metrics["expiry"].dropna().unique()):
    legs = metrics[metrics["expiry"] == exp]
    if legs.empty:
        continue

    spot_val  = float(legs["spot"].iloc[0])
    S_range   = np.linspace(spot_val * 0.6, spot_val * 1.5, 500)
    expiry_pnl = np.zeros_like(S_range)
    today_pnl  = np.zeros_like(S_range)

    for _, leg in legs.iterrows():
        K       = float(leg["strike"])
        prem    = float(leg["lastPrice"]) if pd.notna(leg.get("lastPrice")) else 0.0
        qty     = float(leg.get("contracts", 1)) * CONTRACT_SIZE
        T_leg   = max(float(leg["T"]), 1e-6)
        sigma   = float(leg["implied_vol"]) if pd.notna(leg.get("implied_vol")) else 0.0
        r       = float(leg.get("r", DEFAULT_RF))
        q       = float(leg.get("div_yield", 0.0))
        is_call = leg["option_type"] == "C"

        # Expiry intrinsic payoff
        if is_call:
            expiry_pnl += qty * (np.maximum(S_range - K, 0) - prem)
        else:
            expiry_pnl += qty * (np.maximum(K - S_range, 0) - prem)

        # Today's theoretical P&L at each spot (BS price – entry premium)
        bs_vals    = _bs_vec(is_call, S_range, K, T_leg, r, q, sigma)
        today_pnl += qty * (bs_vals - prem)

    # Max loss / break-even summary
    be_points = S_range[np.where(np.diff(np.sign(expiry_pnl)))[0]]

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(S_range, expiry_pnl, lw=2,   color="navy",      label="At expiry")
    ax.plot(S_range, today_pnl,  lw=2,   color="steelblue", linestyle="--", label="Today (BS)")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(spot_val, color="grey", lw=1, linestyle="--", alpha=0.7, label=f"Spot {spot_val:.1f}")
    ax.fill_between(S_range, expiry_pnl, 0, where=(expiry_pnl > 0), alpha=0.15, color="green", label="Profit zone")
    ax.fill_between(S_range, expiry_pnl, 0, where=(expiry_pnl < 0), alpha=0.15, color="red",   label="Loss zone")
    for _, leg in legs.iterrows():
        ax.axvline(leg["strike"], color="purple", lw=0.8, linestyle=":", alpha=0.6)
    for be in be_points:
        ax.axvline(be, color="orange", lw=1, linestyle="-.", alpha=0.8)
        ax.text(be, ax.get_ylim()[0] * 0.9, f"BE {be:.1f}", fontsize=7, color="orange", ha="center")

    max_loss   = expiry_pnl.min()
    max_profit = expiry_pnl.max()
    title_extra = f"  │  Max Loss {max_loss:,.0f}  │  Max Profit {'∞' if max_profit > 1e6 else f'{max_profit:,.0f}'}"
    ax.set_xlabel("Underlying at expiry")
    ax.set_ylabel("P&L ($)")
    ax.set_title(f"Combined payoff — expiry {pd.Timestamp(exp).strftime('%d %b %Y')}{title_extra}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
## 9 · Scenario analysis — P&L and Greeks at spot moves (IBKR-style)

SCENARIO_MOVES = [-0.20, -0.10, 0.0, 0.10, 0.20, 0.30]

for exp in sorted(metrics["expiry"].dropna().unique()):
    legs = metrics[metrics["expiry"] == exp]
    if legs.empty:
        continue

    spot_val = float(legs["spot"].iloc[0])
    rows = []

    for pct in SCENARIO_MOVES:
        S_scen      = spot_val * (1 + pct)
        pnl_total   = 0.0
        delta_total = 0.0
        gamma_total = 0.0
        vega_total  = 0.0

        for _, leg in legs.iterrows():
            K       = float(leg["strike"])
            prem    = float(leg["lastPrice"]) if pd.notna(leg.get("lastPrice")) else 0.0
            qty     = float(leg.get("contracts", 1)) * CONTRACT_SIZE
            T_leg   = max(float(leg["T"]), 1e-6)
            sigma   = float(leg["implied_vol"]) if pd.notna(leg.get("implied_vol")) else 0.0
            r       = float(leg.get("r", DEFAULT_RF))
            q       = float(leg.get("div_yield", 0.0))
            is_call = leg["option_type"] == "C"

            F_scen     = S_scen * np.exp((r - q) * T_leg)
            D          = np.exp(-r * T_leg)
            price_scen = bs_price_forward(is_call, F_scen, K, sigma, T_leg, D)
            greeks     = bs_greeks_full(leg["option_type"], S_scen, K, T_leg, r, q, sigma)

            pnl_total   += qty * (price_scen - prem)
            delta_total += qty * greeks.get("delta", 0.0)
            gamma_total += qty * greeks.get("gamma", 0.0)
            vega_total  += qty * greeks.get("vega",  0.0)

        rows.append({
            "Move":        f"{pct:+.0%}",
            "Underlying":  round(S_scen, 2),
            "P&L ($)":     round(pnl_total, 2),
            "Delta":       round(delta_total, 3),
            "Gamma":       round(gamma_total, 5),
            "Vega ($)":    round(vega_total, 2),
        })

    df_scen = pd.DataFrame(rows)
    exp_label = pd.Timestamp(exp).strftime("%d %b %Y")
    print(f"\nScenario analysis — expiry {exp_label}")

    def _colour_pnl(v):
        if isinstance(v, (int, float)) and not pd.isna(v):
            return "color: green; font-weight: bold" if v > 0 else "color: red; font-weight: bold" if v < 0 else ""
        return ""

    display(
        df_scen.style
        .map(_colour_pnl, subset=["P&L ($)", "Delta", "Vega ($)"])
        .format({"Underlying": "{:.2f}", "P&L ($)": "{:,.2f}",
                 "Delta": "{:.3f}", "Gamma": "{:.5f}", "Vega ($)": "{:.2f}"})
        .set_properties(**{"text-align": "right"})
        .hide(axis="index")
    )

In [ ]:
## 10 · Historical Vol vs Implied Vol

import yfinance as _yf

HV_WINDOWS = {"HV 20d": 20, "HV 30d": 30, "HV 60d": 60}

hv_rows = []
for tkr in tickers_needed:
    try:
        hist   = _yf.Ticker(tkr).history(period="1y")["Close"]
        rets   = np.log(hist / hist.shift(1)).dropna()

        tkr_m  = metrics[metrics["symbol"].str.upper() == tkr]
        if tkr_m.empty:
            continue
        spot   = float(tkr_m["spot"].iloc[0])

        # ATM-closest IV
        atm_idx = (tkr_m["strike"] - spot).abs().idxmin()
        iv_atm  = float(tkr_m.loc[atm_idx, "implied_vol"])

        row = {"Ticker": tkr, "Spot": round(spot, 2)}
        for label, w in HV_WINDOWS.items():
            hv = float(rets.tail(w).std() * np.sqrt(252))
            row[f"{label} (%)"] = round(hv * 100, 1)
        row["IV ATM (%)"]  = round(iv_atm * 100, 1)
        row["IV/HV 30d"]   = round(iv_atm / (rets.tail(30).std() * np.sqrt(252)), 2)
        hv_rows.append(row)

    except Exception as e:
        print(f"  {tkr}: HV fetch failed — {e}")

if hv_rows:
    df_hv = pd.DataFrame(hv_rows)

    def _colour_ratio(v):
        if isinstance(v, float):
            if v > 1.2:  return "color: red"    # IV expensive vs HV
            if v < 0.8:  return "color: green"  # IV cheap vs HV
        return ""

    display(
        df_hv.style
        .map(_colour_ratio, subset=["IV/HV 30d"])
        .format({c: "{:.1f}" for c in df_hv.columns if "%" in c} |
                {"Spot": "{:.2f}", "IV/HV 30d": "{:.2f}"})
        .set_caption("IV/HV > 1.2 → IV expensive (red)  |  IV/HV < 0.8 → IV cheap (green)")
        .hide(axis="index")
    )

In [ ]:
## 8 · Volatility smile — SVI fitted curve with portfolio strikes marked

for tkr in tickers_needed:
    chain = chain_cache.get(tkr, pd.DataFrame())
    if chain.empty:
        print(f"No market chain for {tkr} — skipping smile chart.")
        continue

    T_vals = chain["T"].dropna().unique() if "T" in chain.columns else []
    if len(T_vals) == 0:
        continue

    spot_val = float(chain["underlying_price"].iloc[0])
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.cm.viridis(np.linspace(0, 1, min(len(T_vals), 10)))

    for t_val, color in zip(sorted(T_vals)[:10], cmap):
        sub = chain[np.isclose(chain["T"], t_val, atol=1e-3)]
        F = float(sub["forward"].iloc[0]) if "forward" in sub.columns else spot_val

        # OTM options for SVI fit (calls above F, puts below F)
        calls_otm = sub[(sub["option_type"] == "C") & (sub["strike"] >= F)]
        puts_otm  = sub[(sub["option_type"] == "P") & (sub["strike"] <  F)]
        otm = pd.concat([puts_otm, calls_otm]).dropna(subset=["implied_vol"]).sort_values("strike")

        label = f"T≈{t_val * 365:.0f}d"

        if len(otm) >= 5:
            params = fit_svi_slice(otm["strike"].values, otm["implied_vol"].values, F, t_val)
            if params is not None:
                # Plot SVI curve over a wide strike range (extrapolates smoothly)
                K_lo = min(otm["strike"].min(), spot_val * 0.60)
                K_hi = max(otm["strike"].max(), spot_val * 1.60)
                K_fine = np.linspace(K_lo, K_hi, 400)
                iv_fine = eval_svi_iv(params, K_fine, F, t_val) * 100
                ax.plot(K_fine, iv_fine, color=color, lw=1.8, alpha=0.8, label=label)
                # Raw liquid data as small dots
                ax.scatter(otm["strike"], otm["implied_vol"] * 100,
                           color=color, s=12, alpha=0.5, zorder=3)
                continue

        # Fallback: just plot raw data as a line
        if len(otm) > 1:
            ax.plot(otm["strike"], otm["implied_vol"] * 100,
                    color=color, lw=1.2, alpha=0.5, linestyle="--", label=label)

    # Mark portfolio strikes for this ticker
    pf_tkr = metrics[metrics["symbol"].str.upper() == tkr]
    y_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 80
    for _, leg in pf_tkr.iterrows():
        ax.axvline(leg["strike"], color="red", lw=1.2, linestyle="--", alpha=0.8)
        ax.text(leg["strike"], y_top,
                f'{leg["option_type"]}K{leg["strike"]:.0f}',
                rotation=90, fontsize=7, color="red", va="top")

    ax.axvline(spot_val, color="black", lw=1.5, linestyle="--", alpha=0.5, label="Spot")
    ax.set_title(f"{tkr} — Implied Vol Smile  (SVI fit · dots = liquid strikes · red = portfolio)")
    ax.set_xlabel("Strike")
    ax.set_ylabel("IV (%)")
    ax.legend(fontsize=7, ncol=4, loc="upper right")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()